# BFI v2 Phase 9 - Formula & Solution Explorer notebook
Every formula: problem -> formula -> code -> output -> limitations. Existing formulas call the real Phase 7/8 code; new ones are marked NEW.

In [1]:
# Step 1: make the Phase 9 package importable (run from the project folder)
import sys, os
os.chdir(os.environ.get('BFI_P9_DIR', '/content/BFI_v2_Phase9_Fraud_Intelligence_Dashboard')); sys.path.insert(0, '.')
from bfi9 import formulas as FX, phase8
print(len(FX.CATALOGUE), 'formulas')

42 formulas


## Data Quality

### Missing-value rate  
*Origin: new - bfi9/quality.py kpis(); v1 notebook cell 13 (missingness report)*

**Problem.** Find incomplete fields before trusting charts or models.  
**Why.** Silent gaps bias statistics and model inputs.

$$\text{Missing\%}=\frac{\#\text{missing cells}}{n\cdot k}\times100$$

`n` = rows; `k` = columns

**Limitations.** Says how much is missing, not why (structural vs random).  
**Interview.** Count missing cells over all cells; then ask whether missingness is structural, like loan_type when has_loan = 0.

In [2]:
# Implementation used by the project:
# missing_pct = 100 * df.isna().sum().sum() / df.size  # df.size = rows x columns
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'missing_rate')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'missing': 377134, 'n': 550000, 'k': 1}
output: 68.56981818181818


### Duplicate rate  
*Origin: historical - v1 notebook cells 8 & 26 ('Duplicate rows: 0')*

**Problem.** Repeated records inflate counts and leak between splits.  
**Why.** Duplicates double-count transactions and evaluation rows.

$$\text{Dup\%}=\frac{\#\text{duplicate rows}}{n}\times100$$

`n` = rows

**Limitations.** Exact duplicates only; near-duplicates need key-based rules.  
**Interview.** df.duplicated() flags exact repeats after the first.

In [3]:
# Implementation used by the project:
# dup_pct = 100 * df.duplicated().sum() / len(df)
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'duplicate_rate')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'duplicates': 0, 'n': 550000}
output: 0.0


### Fraud-label prevalence  
*Origin: existing - Phase 7 metrics_engine.classification_metrics ('prevalence'); v1 Phase 2 report*

**Problem.** Measure class imbalance.  
**Why.** Accuracy looks great when positives are rare; prevalence sets the no-skill AP.

$$\pi=\frac{TP+FN}{N}$$

`π` = share of positive EVALUATION labels (chargeback-derived), not proven crime

**Limitations.** Label-defined; differs between datasets and periods.  
**Interview.** Prevalence is the base rate; AP of a random ranker equals it.

In [4]:
# Implementation used by the project:
# m = metrics_engine.classification_metrics(counts); m['prevalence']
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'prevalence')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))
# Step: the real project value (test split, model A), via Phase 8
print('project value:', f['project']())

inputs: {'TP': 30, 'FP': 20, 'TN': 940, 'FN': 10}


output: 0.04


project value: 0.03476256937907754


### Standardization (z-score)  
*Origin: new - Teaching only - NOT used by the frozen LightGBM pipeline (Phase 4 config: raw numeric inputs)*

**Problem.** Put features on a common scale.  
**Why.** Distance-based models are dominated by large-unit features without it.

$$z=\frac{x-\mu}{\sigma}$$

`μ` = mean; `σ` = standard deviation

**Limitations.** Trees don't need it; fitting μ, σ on all data leaks test information.  
**Interview.** Standardize with training statistics only; trees are scale-invariant.

In [5]:
# Implementation used by the project:
# z = (x - x.mean()) / x.std()  # fit mean/std on TRAINING data only
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'zscore')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'x': 250.0, 'mu': 120.0, 'sigma': 60.0}
output: 2.1666666666666665


### IQR outlier rule  
*Origin: historical - v1 notebook Step 10; bfi9/quality.outliers()*

**Problem.** Flag extreme values.  
**Why.** Extreme amounts can distort means and models.

$$x<Q_1-1.5\,IQR\ \lor\ x>Q_3+1.5\,IQR,\quad IQR=Q_3-Q_1$$

`Q1, Q3` = 25th/75th percentiles

**Limitations.** Skewed data flags many 'outliers'; bounds from all rows leak future data.  
**Interview.** IQR is robust to the extremes it is detecting, unlike mean ± 3σ.

In [6]:
# Implementation used by the project:
# q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
# is_out = (s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'iqr_outlier')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'x': 10000.0, 'q1': 763.04, 'q3': 8557.5}
output: {'lower': -10928.650000000001, 'upper': 20249.190000000002, 'outlier': False}


### Cyclic hour encoding (v1 Phase 3)  
*Origin: historical - v1 notebook Step 11 (hour_sin, hour_cos)*

**Problem.** Make 23:00 close to 00:00.  
**Why.** Raw hour treats midnight neighbours as far apart.

$$(\sin\tfrac{2\pi h}{24},\ \cos\tfrac{2\pi h}{24})$$

`h` = hour 0–23

**Limitations.** Needs both sin and cos to be unambiguous.  
**Interview.** Maps the clock onto a circle.

In [7]:
# Implementation used by the project:
# df['hour_sin'] = np.sin(2*np.pi*df.transaction_hour/24)
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'cyclic_hour')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'hour': 23}
output: {'sin': -0.25881904510252157, 'cos': 0.9659258262890681}


## Classification

### Score → decision  
*Origin: existing - Phase 4 BFI_v2_model_config.json threshold; Phase 8 data_access.predictions()*

**Problem.** Turn a score into an alert.  
**Why.** A score alone does not tell an analyst what to do.

$$\hat y_i=\mathbb 1[s_i\ge t]$$

`s` = uncalibrated model score; `t` = frozen threshold (read from Phase 4 config)

**Limitations.** Raising t cuts FP and raises FN; t must not be re-tuned on the spent test set.  
**Interview.** The inclusive rule s ≥ t; the threshold controls alert volume, not truth.

In [8]:
# Implementation used by the project:
# decision = np.where(score >= t, 'ALERT (review)', 'no alert')
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'threshold_rule')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'score': 0.2, 'threshold': 0.12379763346033115}
output: ALERT (review)


### Class-weight ratio (v1 Phase 2)  
*Origin: historical - v1 Phase 2 report §11 'Class-weight ratio'; BFI_Phase2_Summary.csv*

**Problem.** Counter class imbalance in training.  
**Why.** Without weighting the learner can ignore rare positives.

$$w=\frac{\#\text{negatives}}{\#\text{positives}}$$



**Limitations.** Reweighting distorts score calibration.  
**Interview.** v1 used ≈111.9:1 as LightGBM scale_pos_weight.

In [9]:
# Implementation used by the project:
# scale_pos_weight = (y == 0).sum() / (y == 1).sum()
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'class_weight')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'neg': 545127, 'pos': 4873}
output: 111.86681715575621


## Evaluation Metrics

### Precision  
*Origin: existing - Phase 7 metrics_engine.classification_metrics*

**Problem.** How many alerts carry a positive label?  
**Why.** Review effort is wasted on false alerts.

$$\frac{TP}{TP+FP}$$

`TP` = alert ∧ label 1; `FP` = alert ∧ label 0

**Limitations.** Undefined (NaN) with no alerts; depends on prevalence.  
**Interview.** Of what we flagged, how much was right.

In [10]:
# Implementation used by the project:
# m = metrics_engine.classification_metrics(counts); m['precision']
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'precision')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))
# Step: the real project value (test split, model A), via Phase 8
print('project value:', f['project']())

inputs: {'TP': 30, 'FP': 20, 'TN': 940, 'FN': 10}
output: 0.6


project value: 0.4901287553648069


### Recall (TPR)  
*Origin: existing - Phase 7 metrics_engine.classification_metrics*

**Problem.** How many positive labels were caught?  
**Why.** Missed fraud is the costliest error.

$$\frac{TP}{TP+FN}$$

`FN` = no alert ∧ label 1

**Limitations.** Ignores false alerts; trivially 1 if everything is flagged.  
**Interview.** Of the positives, how many we caught.

In [11]:
# Implementation used by the project:
# m = metrics_engine.classification_metrics(counts); m['recall']
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'recall')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))
# Step: the real project value (test split, model A), via Phase 8
print('project value:', f['project']())

inputs: {'TP': 30, 'FP': 20, 'TN': 940, 'FN': 10}
output: 0.75


project value: 0.5331465919701214


### Specificity (TNR)  
*Origin: existing - Phase 7 metrics_engine.classification_metrics*

**Problem.** How many negatives were left alone?  
**Why.** Customer friction comes from flagged legitimate payments.

$$\frac{TN}{TN+FP}$$

`TN` = no alert ∧ label 0

**Limitations.** Near 1 whenever negatives dominate.  
**Interview.** Share of legitimate transactions not flagged.

In [12]:
# Implementation used by the project:
# m = metrics_engine.classification_metrics(counts); m['specificity']
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'specificity')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))
# Step: the real project value (test split, model A), via Phase 8
print('project value:', f['project']())

inputs: {'TP': 30, 'FP': 20, 'TN': 940, 'FN': 10}
output: 0.9791666666666666


project value: 0.9800255565270025


### F1 score  
*Origin: existing - Phase 7 metrics_engine.classification_metrics*

**Problem.** One number balancing precision and recall.  
**Why.** Optimising one alone is easy and useless.

$$2\frac{PR}{P+R}$$

`P` = precision; `R` = recall

**Limitations.** Ignores TN and costs; equal weights may not match the business.  
**Interview.** Harmonic mean punishes imbalance between P and R.

In [13]:
# Implementation used by the project:
# m = metrics_engine.classification_metrics(counts); m['f1']
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'f1')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))
# Step: the real project value (test split, model A), via Phase 8
print('project value:', f['project']())

inputs: {'TP': 30, 'FP': 20, 'TN': 940, 'FN': 10}
output: 0.6666666666666665


project value: 0.5107334525939177


### Accuracy  
*Origin: existing - Phase 7 metrics_engine.classification_metrics*

**Problem.** Share of correct decisions.  
**Why.** Shows why it misleads under imbalance.

$$\frac{TP+TN}{N}$$

`N` = all transactions

**Limitations.** A never-alert model scores 1 − π.  
**Interview.** With 3.5% positives, 96.5% accuracy is free.

In [14]:
# Implementation used by the project:
# m = metrics_engine.classification_metrics(counts); m['accuracy']
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'accuracy')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))
# Step: the real project value (test split, model A), via Phase 8
print('project value:', f['project']())

inputs: {'TP': 30, 'FP': 20, 'TN': 940, 'FN': 10}
output: 0.97


project value: 0.9644908955175435


### Balanced accuracy  
*Origin: existing - Phase 7 metrics_engine.classification_metrics*

**Problem.** Accuracy that weights both classes equally.  
**Why.** Removes the majority-class illusion.

$$\frac{TPR+TNR}{2}$$



**Limitations.** Ignores precision.  
**Interview.** Average of recall and specificity.

In [15]:
# Implementation used by the project:
# m = metrics_engine.classification_metrics(counts); m['balanced_accuracy']
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'balanced_accuracy')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))
# Step: the real project value (test split, model A), via Phase 8
print('project value:', f['project']())

inputs: {'TP': 30, 'FP': 20, 'TN': 940, 'FN': 10}
output: 0.8645833333333333


project value: 0.756586074248562


### False-positive rate  
*Origin: existing - Phase 7 metrics_engine.classification_metrics*

**Problem.** Share of negatives wrongly flagged.  
**Why.** Drives customer friction and ROC.

$$\frac{FP}{FP+TN}$$



**Limitations.** Small FPR can still mean many FPs.  
**Interview.** 1 − specificity.

In [16]:
# Implementation used by the project:
# m = metrics_engine.classification_metrics(counts); m['fpr']
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'fpr')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))
# Step: the real project value (test split, model A), via Phase 8
print('project value:', f['project']())

inputs: {'TP': 30, 'FP': 20, 'TN': 940, 'FN': 10}
output: 0.020833333333333332


project value: 0.01997444347299751


### False-negative rate  
*Origin: existing - Phase 7 metrics_engine.classification_metrics*

**Problem.** Share of positives missed.  
**Why.** Missed-fraud exposure.

$$\frac{FN}{FN+TP}$$



**Limitations.** Label-defined misses only.  
**Interview.** 1 − recall.

In [17]:
# Implementation used by the project:
# m = metrics_engine.classification_metrics(counts); m['fnr']
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'fnr')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))
# Step: the real project value (test split, model A), via Phase 8
print('project value:', f['project']())

inputs: {'TP': 30, 'FP': 20, 'TN': 940, 'FN': 10}
output: 0.25


project value: 0.4668534080298786


### False discovery rate  
*Origin: new - bfi9/formulas.fdr (= 1 − Phase 7 precision; tested)*

**Problem.** Share of alerts that are false.  
**Why.** Tells reviewers how much of their queue is noise.

$$FDR=\frac{FP}{TP+FP}=1-\text{Precision}$$



**Limitations.** Undefined with no alerts.  
**Interview.** FDR is the complement of precision.

In [18]:
# Implementation used by the project:
# fdr = FP / (TP + FP) if (TP + FP) else float('nan')
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'fdr')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))
# Step: the real project value (test split, model A), via Phase 8
print('project value:', f['project']())

inputs: {'TP': 30, 'FP': 20}
output: 0.4


project value: 0.5098712446351932


### Matthews correlation coefficient  
*Origin: new - bfi9/formulas.mcc*

**Problem.** One balanced score using all four cells.  
**Why.** Robust summary for imbalanced binary problems.

$$\frac{TP\,TN-FP\,FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

`range` = −1 … 1; 0 = no better than chance

**Limitations.** Zero marginals make it undefined (NaN, not 0).  
**Interview.** A correlation between predictions and labels; 0 means chance-level.

In [19]:
# Implementation used by the project:
# den = sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN)); mcc = (TP*TN - FP*FN)/den if den else nan
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'mcc')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'TP': 30, 'FP': 20, 'TN': 940, 'FN': 10}
output: 0.6556100681071858


### Average precision  
*Origin: existing - Phase 7 metrics_engine.ranking_metrics (sklearn average_precision_score)*

**Problem.** Rank quality when positives are rare.  
**Why.** Threshold-free summary focused on the positive class.

$$AP=\sum_k (R_k-R_{k-1})P_k$$

`P_k, R_k` = precision/recall at the k-th threshold

**Limitations.** Step-wise (no interpolation); depends on prevalence.  
**Interview.** Area under the step PR curve; compare to π.

In [20]:
# Implementation used by the project:
# ap = metrics_engine.ranking_metrics(y, s)['average_precision']
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'average_precision')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))
# Step: the real project value (test split, model A), via Phase 8
print('project value:', f['project']())

inputs: {'prevalence': 0.0348}
output: {'no_skill_AP': 0.0348}
project value: 0.5506197344626254


### ROC-AUC  
*Origin: existing - Phase 7 metrics_engine.ranking_metrics (sklearn roc_auc_score)*

**Problem.** Ranking discrimination.  
**Why.** Threshold-free view of separation.

$$AUC=P(s_+>s_-)$$



**Limitations.** Not calibration, not profit; can look high when precision is low.  
**Interview.** Chance a random positive outranks a random negative.

In [21]:
# Implementation used by the project:
# auc = metrics_engine.ranking_metrics(y, s)['roc_auc']
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'roc_auc')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))
# Step: the real project value (test split, model A), via Phase 8
print('project value:', f['project']())

inputs:

 {'pairs_correct': 905, 'pairs_total': 1000}
output: 0.905


project value: 0.9051527755720846


### Relative AP improvement (v1 Phase 2)  
*Origin: historical - v1 Phase 2 report §11; summary CSV (≈13.0%)*

**Problem.** Compare two models' AP.  
**Why.** Absolute AP differences look tiny at low prevalence.

$$\frac{AP_{new}-AP_{old}}{AP_{old}}\times100$$



**Limitations.** Not computable from a zero baseline.  
**Interview.** v1 reduced model: +13% relative AP, still near no-skill.

In [22]:
# Implementation used by the project:
# 100 * (ap_new - ap_old) / ap_old
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'relative_ap')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'ap_old': 0.015473, 'ap_new': 0.01749}
output: 13.035610418147728


## Costs & Decisions

### Total illustrative cost (Phase 7)  
*Origin: existing - Phase 7 cost_engine.simple_total_cost / cost_from_outcomes*

**Problem.** Translate errors into an assumed cost.  
**Why.** Metrics ignore that misses and false alerts cost different amounts.

$$C=FP\,C_{FP}+FN\,C_{FN}+(TP+FP)\,C_{rev}+TP\,C_{int}$$

`C_FP` = friction per false alert (excl. review); `C_FN` = loss per missed fraud; `C_rev` = per review

**Limitations.** Illustrative units (CU), not money; assumes every alert reviewed.  
**Interview.** Weighted error count; the weights are business assumptions.

In [23]:
# Implementation used by the project:
# cost_engine.simple_total_cost(TP, FP, FN, CostConfig(name, c_fp=5, c_fn=100, c_review=2))
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'total_cost')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'TP': 1713, 'FP': 1782, 'FN': 1500, 'c_fp': 5.0, 'c_fn': 100.0, 'c_review': 2.0}
output: 165900.0


### Break-even missed-fraud cost  
*Origin: existing - Phase 7 sensitivity_analysis.break_even_fn_cost*

**Problem.** When is screening worth it?  
**Why.** Shows how high C_FN must be before alerts pay for themselves.

$$C_{FN}^*=\frac{A}{\text{caught}}$$

`A` = all non-FN costs

**Limitations.** Depends on the other assumed costs.  
**Interview.** Cost per caught fraud at which screening breaks even.

In [24]:
# Implementation used by the project:
# cost_adapter.break_even('illustrative_baseline')
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'break_even')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'other_costs': 15900.0, 'caught': 1713}
output: 9.281961471103328


### Expected-cost (Bayes) threshold  
*Origin: existing - Phase 7 cost_engine.bayes_threshold*

**Problem.** Cost-optimal cut-off IF scores were calibrated.  
**Why.** Links costs to a threshold.

$$p^*=\frac{C_{rev}+C_{FP}}{C_{FN}+C_{FP}-C_{int}}$$



**Limitations.** Invalid without calibration (not established).  
**Interview.** Flag when expected loss of passing exceeds expected cost of acting.

In [25]:
# Implementation used by the project:
# cost_engine.bayes_threshold(cfg)
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'bayes_threshold')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'c_fp': 5.0, 'c_fn': 100.0, 'c_review': 2.0}
output: 0.06666666666666667


### Implied C_FN / C_FP  
*Origin: existing - Phase 7 cost_engine.implied_fn_fp_ratio*

**Problem.** What cost ratio does the frozen threshold imply?  
**Why.** Makes the hidden assumption explicit.

$$r=\frac1t-1$$



**Limitations.** Assumes calibration and no review cost.  
**Interview.** Threshold t is optimal when C_FN/C_FP = 1/t − 1.

In [26]:
# Implementation used by the project:
# cost_engine.implied_fn_fp_ratio(t)
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'implied_ratio')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'threshold': 0.12379763346033115}
output: 7.077698838406576


## Risk Bands

### Risk-band mapping  
*Origin: existing - Phase 5 risk_band_config.json; bfi_phase5.risk_band (lower bound inclusive)*

**Problem.** Prioritise alerts.  
**Why.** Four bands are easier to triage than raw scores.

$$\text{band}(s)=\max\{b_j: s\ge \text{min}_j\}$$

`min_j` = band lower bounds: 0, t/2, t, 0.5

**Limitations.** PROVISIONAL; bands are not probabilities.  
**Interview.** Bands are ordinal score ranges for triage, not calibrated risk levels.

In [27]:
# Implementation used by the project:
# band = bfi_phase5.risk_band(score, band_config)
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'risk_band')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'score': 0.3, 'threshold': 0.12379763346033115}
output: High


## Statistics & Probability

### Wilson interval (proportion)  
*Origin: existing - Phase 7 metrics_engine.wilson_interval*

**Problem.** Uncertainty for a rate.  
**Why.** A normal interval misbehaves near 0/1 and small n.

$$\frac{\hat p+\frac{z^2}{2n}\pm z\sqrt{\frac{\hat p(1-\hat p)}{n}+\frac{z^2}{4n^2}}}{1+\frac{z^2}{n}}$$

`z` = 1.96 for 95%

**Limitations.** Assumes independent rows; clustered fraud makes it too narrow.  
**Interview.** Score interval that stays inside [0,1] and works for small n.

In [28]:
# Implementation used by the project:
# lo, hi = metrics_engine.wilson_interval(k, n, 0.95)
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'wilson')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'k': 1713, 'n': 3495}
output: (np.float64(0.4735753574711492), np.float64(0.5067038289932593))


### Brier score  
*Origin: existing - Phase 7 metrics_engine.brier_score*

**Problem.** Probability-style error of scores.  
**Why.** Mixes calibration and discrimination.

$$\frac1n\sum (s_i-y_i)^2$$



**Limitations.** Low Brier does not prove calibration.  
**Interview.** Mean squared error of the score against the 0/1 label.

In [29]:
# Implementation used by the project:
# metrics_engine.brier_score(y, s)
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'brier')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'s': 0.8, 'y': 1}
output: 0.03999999999999998


### Mean, sample variance, std  
*Origin: existing - Phase 8 stats_teaching.describe (ddof=1)*

**Problem.** Summarise a numeric field.  
**Why.** Centre and spread drive every comparison.

$$\bar x=\tfrac1n\sum x_i,\ s^2=\tfrac1{n-1}\sum(x_i-\bar x)^2$$

`n−1` = Bessel's correction

**Limitations.** Skewed scores: median is more typical than the mean.  
**Interview.** n−1 makes the variance unbiased.

In [30]:
# Implementation used by the project:
# stats_teaching.describe(values)
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'mean_var')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'a': 2.0, 'b': 4.0, 'c': 9.0}
output: {'mean': np.float64(5.0), 'var': np.float64(13.0), 'std': np.float64(3.605551275463989)}


### t confidence interval (mean)  
*Origin: new - bfi9/formulas.t_ci_mean*

**Problem.** Uncertainty of a mean.  
**Why.** A point estimate hides sampling error.

$$\bar x\pm t_{1-\alpha/2,n-1}\frac{s}{\sqrt n}$$



**Limitations.** Independence assumed; prefer cluster bootstrap for transactions.  
**Interview.** Mean ± t-quantile × standard error.

In [31]:
# Implementation used by the project:
# t_ci_mean(values, 0.95)
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 't_ci')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'a': 2.0, 'b': 4.0, 'c': 9.0}
output: (np.float64(-3.9566858949186035), np.float64(13.956685894918603))


### Conditional probability by band  
*Origin: existing - Phase 8 stats_teaching.band_table (evaluation mode)*

**Problem.** Label rate inside a group.  
**Why.** Turns counts into group-level rates.

$$P(A\mid B)=\frac{P(A\cap B)}{P(B)}$$



**Limitations.** Small groups give unstable rates.  
**Interview.** Rate among the group, not among everyone.

In [32]:
# Implementation used by the project:
# t['fraud_labels'] / t['transactions']
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'conditional')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'both': 150, 'group': 1000}
output: 0.15


### Bayes' theorem (alert)  
*Origin: existing - Phase 8 metrics_adapter.bayes_view*

**Problem.** Probability-like reading of an alert.  
**Why.** Shows why rare fraud makes false alerts common.

$$P(F\mid A)=\frac{TPR\,\pi}{TPR\,\pi+FPR(1-\pi)}$$



**Limitations.** Population-level; not the probability for one live alert.  
**Interview.** Equals precision on the same split (a built-in check).

In [33]:
# Implementation used by the project:
# metrics_adapter.bayes_view('test')
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'bayes')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs:

 {'tpr': 0.533, 'fpr': 0.02, 'pi': 0.0348}
output: 0.4900191269245807


### Pearson correlation  
*Origin: historical - v1 notebook Step 12 (target correlation); bfi9/quality.correlations*

**Problem.** Linear association between two numeric fields.  
**Why.** Screens redundant or related features.

$$r=\frac{\sum(x-\bar x)(y-\bar y)}{\sqrt{\sum(x-\bar x)^2\sum(y-\bar y)^2}}$$

`range` = −1 … 1

**Limitations.** Linear only; correlation is not causation.  
**Interview.** v1's top |r| with is_fraud was 0.06: almost no linear signal.

In [34]:
# Implementation used by the project:
# df[cols].corr(method='pearson')
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'pearson')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs:

 {'x1': 1.0, 'x2': 2.0, 'x3': 3.0, 'y1': 2.0, 'y2': 4.1, 'y3': 5.9}
output: 0.999015263178192


## Explainability

### TreeSHAP additivity (log-odds)  
*Origin: existing - Phase 5 bfi_phase5 (LightGBM pred_contrib); Phase 8 explain()*

**Problem.** Split one score into feature contributions.  
**Why.** Investigators need reasons, not just a number.

$$f(x)=\phi_0+\sum_j\phi_j,\quad s=\sigma(f(x))$$

`f(x)` = raw log-odds; `φ0` = base value; `σ` = logistic function

**Limitations.** Correlated features share credit; not causal; says nothing about availability.  
**Interview.** Contributions add up exactly to the model's log-odds.

In [35]:
# Implementation used by the project:
# contrib = booster.predict(X, pred_contrib=True)  # last column = base
# score = 1/(1+exp(-contrib.sum(axis=1)))
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'shap_additivity')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs:

 {'base': -3.2, 'sum_contrib': 2.5}
output: {'log_odds': -0.7000000000000002, 'score': 0.33181222783183384}


## Global Scam Intelligence

### Percentage change (zero-safe)  
*Origin: new - bfi9/quality.pct_change*

**Problem.** Change in reported incidents.  
**Why.** Raw counts hide relative growth.

$$\%\Delta=\frac{N_t-N_{t-1}}{N_{t-1}}\times100$$



**Limitations.** More reports ≠ more fraud (coverage changes).  
**Interview.** Never divide by a zero baseline; say so instead.

In [36]:
# Implementation used by the project:
# pct_change(prev, curr)  # None when prev == 0
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'pct_change')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'prev': 0.0, 'curr': 4.0}
output: Not calculable from zero baseline


## Stock-Market Impact

### Simple return  
*Origin: new - bfi9/formulas.simple_return; intelligence/event_study.py*

**Problem.** Daily price change.  
**Why.** Comparable across price levels.

$$R_t=\frac{P_t}{P_{t-1}}-1$$



**Limitations.** Adjust for splits/dividends.  
**Interview.** Relative change in price.

In [37]:
# Implementation used by the project:
# r = close.pct_change()
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'simple_return')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'p0': 100.0, 'p1': 97.0}
output: -0.030000000000000027


### Log return  
*Origin: new - bfi9/formulas.log_return*

**Problem.** Additive multi-day returns.  
**Why.** Sums over time correctly.

$$r_t=\ln\frac{P_t}{P_{t-1}}$$



**Limitations.** Prices must be > 0.  
**Interview.** Log returns add across days.

In [38]:
# Implementation used by the project:
# r = np.log(close).diff()
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'log_return')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'p0': 100.0, 'p1': 97.0}
output: -0.030459207484708574


### Benchmark-adjusted & market-model abnormal return  
*Origin: new - intelligence/event_study.py*

**Problem.** Separate a stock's move from the market's.  
**Why.** A drop on a market-wide down day is not company-specific.

$$AR_t=R_{i,t}-(\hat\alpha+\hat\beta R_{m,t})$$

`α, β` = OLS on the estimation window

**Limitations.** Confounding news; not causal.  
**Interview.** Residual return after the market explains what it can.

In [39]:
# Implementation used by the project:
# ar = r_i - (alpha + beta * r_m)
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'abnormal_return')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'r_i': -0.05, 'r_m': -0.01, 'alpha': 0.0, 'beta': 1.2}
output: -0.038000000000000006


### Cumulative abnormal return  
*Origin: new - intelligence/event_study.py*

**Problem.** Total abnormal move over an event window.  
**Why.** Single days miss slow reactions.

$$CAR[t_1,t_2]=\sum_{t=t_1}^{t_2}AR_t$$



**Limitations.** Window choice matters; association only.  
**Interview.** Sum of abnormal returns over the window.

In [40]:
# Implementation used by the project:
# car = ar.loc[t1:t2].sum()
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'car')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'ar1': -0.01, 'ar2': -0.03, 'ar3': 0.005}
output: -0.035


## Forecasting & Monitoring

### Seasonal-naive (7-day) forecast  
*Origin: new - bfi9/forecasting.predict*

**Problem.** Baseline for daily volume.  
**Why.** Any model must beat a simple baseline.

$$\hat y_{T+h}=y_{T+h-7\lceil h/7\rceil}$$



**Limitations.** Relative days only; no calendar effects.  
**Interview.** Repeat last week's pattern.

In [41]:
# Implementation used by the project:
# forecasting.predict(history, 'seasonal_naive_7', h)
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'seasonal_naive')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'d1': 3000.0, 'd2': 3200.0, 'd3': 3100.0, 'd4': 2900.0, 'd5': 3050.0, 'd6': 3300.0, 'd7': 3400.0}
output: [np.float64(3000.0), np.float64(3200.0), np.float64(3100.0), np.float64(2900.0), np.float64(3050.0), np.float64(3300.0), np.float64(3400.0)]


### MAE / MAPE  
*Origin: new - bfi9/forecasting.backtest*

**Problem.** Forecast error.  
**Why.** Needed to compare methods honestly.

$$MAE=\tfrac1h\sum|y-\hat y|,\ MAPE=\tfrac{100}h\sum\tfrac{|y-\hat y|}{y}$$



**Limitations.** MAPE explodes near zero.  
**Interview.** MAE in units; MAPE in percent.

In [42]:
# Implementation used by the project:
# np.mean(np.abs(err)); np.mean(np.abs(err)/actual)*100
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'mae_mape')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'y': 3000.0, 'yhat': 2850.0}
output: {'AE': 150.0, 'APE_pct': 5.0}


### Population Stability Index  
*Origin: new - bfi9/forecasting.psi*

**Problem.** Detect score-distribution drift.  
**Why.** Shifts can silently degrade a frozen model.

$$PSI=\sum_b (a_b-e_b)\ln\frac{a_b}{e_b}$$

`e, a` = reference / current bin shares

**Limitations.** Rule-of-thumb cut-offs only; bins matter.  
**Interview.** Symmetric KL-style distance between binned distributions.

In [43]:
# Implementation used by the project:
# forecasting.psi(val_scores, test_scores)
# Step: run the worked example with its educational inputs
f = next(x for x in FX.CATALOGUE if x['id'] == 'psi')
print('inputs:', f['inputs'])
print('output:', f['run'](**f['inputs']))

inputs: {'e1': 0.5, 'e2': 0.5, 'a1': 0.6, 'a2': 0.4}
output: 0.04054651081081642
